# Mistake detection benchmark — LadderSym comparison

**Run All** reloads matching completed results in fresh Python subprocesses. The setup table shows the saved run configuration when resuming, or current production defaults for a new run. Cached results retain their recorded code versions; they are not relabeled as a new evaluation of changed production code.

Compare **Attune weighted string editing**, **Parangonar DualDTW / Automatic / TheGlueNote**, and **Nakamura** on exact performed MIDI notes and Attune-extracted notes. The audio comparison also includes production **Attune repeat-only refinement**, legacy **Attune Checker 3**, and the original **PolyTune** and **LadderSym (CocoChorales, prompted)** audio systems.

Run All reuses completed checkpoints and rebuilds the saved tables without detection or neural inference. Cached results retain the algorithm versions recorded in `run.json`, even if local code has changed. Set `FORCE_PITCH_DETECTION = True` only for an intentional fresh experiment in a new output directory. New `_laddersym_2026-09-28` result folders preserve prior runs. Attune bootstraps with an endpoint fit, aligns, robustly fits matched onsets once, re-aligns, and then runs repeat-only refinement. Legacy Checker 3 retains its original iterative flow. Short/long edits stay within 0.5–1.5× each note’s own duration and exceed the shared duration tolerance by at least 50 ms (300 ms minimum with current settings). Ineligible duration edits are recorded as skipped. Cumulative onset drift is retained. The injection condition matches the preceding `_relative_duration` run. Matching case IDs permit a paired algorithm comparison against that run; comparisons against `_promoted` still change the injection condition. This sample was used for tuning; it is not an independent evaluation.

One-time setup in this kernel/environment:
```sh
python -m benchmarks.modules.mistake.competitors.Nakamura setup
python -m benchmarks.modules.mistake.competitors.PolyTune setup
python -m benchmarks.modules.mistake.competitors.LadderSym setup
python -m pip install parangonar==3.3.3 torch symusic==0.5.9 miditok==3.0.6.post1
```
Missing competitors fail explicitly. Setup and scoring details: [README](../modules/mistake/README.md). Promotion evidence: [report](../../docs/alignment-promotion-2026-09-27.md).

**Boundary policy (September 29):** detected notes are no longer corrected using
performed-MIDI first/last durations. Audio-based score timing fits stay enabled.
Earlier 95.3% results used that correction and remain historical; the new output
folder prevents replaying them as current results. The fresh Attune-only audit is
in `benchmarks/results/mistake_competitors_coco_unassisted_2026-09-29`.

Fresh verification without boundary correction: **71 TP / 4 FP / 3 FN = 95.3%**
on the 26 injected cases; **zero clean-control false positives**. All regenerated
MIDI/audio inputs were checked byte-for-byte against the historical run. This
verification reran Attune; the full comparison cells below target a fresh folder.

**Run All caching:** unchanged generated MIDI/audio and compatible pitch tracks are
reused across run folders. PolyTune and LadderSym predictions are reused after input,
prompt, checkpoint and environment checks. Only downstream analysis affected by the
removed boundary correction is recomputed; corrected note caches are never imported.
Leave `FORCE_PITCH_DETECTION=False` for normal reruns.


In [1]:
from pathlib import Path
import sys
import subprocess
import json
import pandas as pd

REPO_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").is_file()
    )
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from IPython.display import display
from benchmarks.modules.mistake.MistakeBenchmarker import COMPETITOR_METHODS
from benchmarks.modules.mistake.datasets.MistakeCases import select_coco_sources
import importlib
from benchmarks.modules.mistake.MistakeNotebook import MistakeNotebook

importlib.reload(ResultOverview)
COCO_ROOT = REPO_ROOT / "benchmarks/datasets/cocochorales_tiny"
OUTPUT = (
    REPO_ROOT / "benchmarks/results/mistake_competitors_coco_unassisted_full_2026-09-29"
)
SYMBOLIC_OUTPUT = (
    REPO_ROOT
    / "benchmarks/results/mistake_competitors_coco_symbolic_laddersym_2026-09-28"
)
PREVIOUS_OUTPUT = (
    REPO_ROOT / "benchmarks/results/mistake_competitors_coco_relative_duration"
)
PER_INSTRUMENT = 1
SAMPLE_SEED = 0
SEEDS = (0, 1)
INJECTION_RATE = 0.25
TOLERANCES = (0.05, 0.1, 0.2)
PRIMARY_TOLERANCE = 0.1
FORCE_PITCH_DETECTION = False
SELECTED_METHODS = COMPETITOR_METHODS
POLYTUNE_DEVICE = "cpu"
LADDERSYM_DEVICE = "cpu"
WORKERS = 8
NEURAL_WORKERS = None
POLYTUNE_LAST = True
RUNNER = [sys.executable, "-m", "benchmarks.modules.mistake.MistakeBenchmarker"]
CURRENT_DEFAULTS = json.loads(
    subprocess.check_output(RUNNER + ["--defaults"], cwd=REPO_ROOT, text=True)
)
saved_defaults = OUTPUT / "production_defaults.json"
DEFAULTS = (
    json.loads(saved_defaults.read_text())
    if not FORCE_PITCH_DETECTION and saved_defaults.exists()
    else CURRENT_DEFAULTS
)
print(
    "Cache reuse enabled; completed results retain their recorded code and settings."
    if not FORCE_PITCH_DETECTION
    else "Fresh detection explicitly requested."
)
DISPLAY_FIELDS = [
    "unv_thresh",
    "min_volume",
    "posthoc_unv_thresh",
    "posthoc_min_volume",
    "min_note_length_cap",
    "min_note_length_factor",
    "pitch_thresh",
    "min_silence_duration_ms",
    "alignment_gamma_pitch",
    "alignment_gamma_time",
    "alignment_alpha_onset",
    "alignment_alpha_duration",
    "ins_cost",
    "del_cost",
    "timing_tolerance",
]
display(
    pd.Series({k: DEFAULTS[k] for k in DISPLAY_FIELDS}, name="Run setting").to_frame()
)
print(
    f"Workers: {WORKERS} CPU, then {NEURAL_WORKERS or 'auto-sized'} persistent audio-model workers"
)
print(
    "Duration factors: 0.5–1.5×; minimum duration error:",
    max(0.3, DEFAULTS["timing_tolerance"] + 0.05),
    "seconds; cumulative drift retained",
)
print("Audio results:", OUTPUT)
print("Symbolic results:", SYMBOLIC_OUTPUT)


def run_stage(stage, output, methods):
    request = dict(
        stage=stage,
        output=str(output),
        sources=sample.source.tolist(),
        source_metadata=sample.set_index("source").to_dict(orient="index"),
        expected_defaults=DEFAULTS,
        seeds=SEEDS,
        rates=(0.0, INJECTION_RATE),
        methods=methods,
        tolerances=TOLERANCES,
        workers=WORKERS,
        neural_workers=NEURAL_WORKERS,
        polytune_last=POLYTUNE_LAST,
        polytune_device=POLYTUNE_DEVICE,
        laddersym_device=LADDERSYM_DEVICE,
        force_pitch_detection=FORCE_PITCH_DETECTION,
    )
    subprocess.run(
        RUNNER, input=json.dumps(request), text=True, cwd=REPO_ROOT, check=True
    )
    metadata = json.loads((output / "run.json").read_text())
    if metadata["status"] != "complete":
        raise RuntimeError(f"Incomplete {stage} run: {output}")
    return pd.read_csv(output / "rows.csv")


Cache reuse enabled; completed results retain their recorded code and settings.


,Run setting
unv_thresh,0.985
min_volume,0.030
posthoc_unv_thresh,0.990
posthoc_min_volume,0.040
min_note_length_cap,0.200
min_note_length_factor,0.500
pitch_thresh,0.600
min_silence_duration_ms,40.000
alignment_gamma_pitch,4.000
alignment_gamma_time,1.000


Workers: 8 CPU, then auto-sized persistent audio-model workers
Duration factors: 0.5–1.5×; minimum duration error: 0.3 seconds; cumulative drift retained
Audio results: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/mistake_competitors_coco_unassisted_full_2026-09-29
Symbolic results: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/mistake_competitors_coco_symbolic_laddersym_2026-09-28


## Paired CocoChorales sample

One test stem per instrument, two seeds, clean and 25% injection conditions. Insertions move subsequent notes later; deletions move them earlier. Performances stay monophonic, with changed pitches on the semitone grid. Net mistakes are recounted after all edits, accounting for those shifts.

In [2]:
sample = select_coco_sources(
    root=COCO_ROOT, split="test", per_instrument=PER_INSTRUMENT, seed=SAMPLE_SEED
)
OUTPUT.mkdir(parents=True, exist_ok=True)
sample.to_csv(OUTPUT / "sample.csv", index=False)
display(sample[["instrument", "track_id", "split"]])
print(
    f"{len(sample)} stems × {len(SEEDS)} seeds × 2 conditions = {len(sample) * len(SEEDS) * 2} shared cases"
)


,instrument,track_id,split
0,trumpet,test__brass_track222970__1_trumpet,test
1,tuba,test__brass_track223079__4_tuba,test
2,trombone,test__brass_track223441__3_trombone,test
3,horn,test__brass_track223724__2_horn,test
4,double bass,test__random_track234028__4_double bass,test
5,saxophone,test__random_track234460__3_saxophone,test
6,violin,test__string_track216129__2_violin,test
7,cello,test__string_track216876__4_cello,test
8,viola,test__string_track217297__3_viola,test
9,clarinet,test__woodwind_track228702__3_clarinet,test


13 stems × 2 seeds × 2 conditions = 52 shared cases


## Symbolic comparison: weighted string editing versus external aligners

Runs on exact notes from the final injected MIDI, preserving performed timing and mistakes. **Attune (no refinement)** runs plain weighted string editing on the original, unmerged notes, with the initial score-span fit. Repeat splitting, same-pitch grouping, Checker 3, and iterative refinement are excluded. External aligners receive the original score. Nakamura includes its original HMM, error detection and realignment stages.

Scores below are pooled **mistake-event** precision/recall/F1, not percentage of correctly aligned notes. Substitutions count as missed + extra. The onset gate is 100 ms and pitch gate 50 cents. Duration errors remain separate. PolyTune takes audio and has no symbolic row.


In [3]:
symbolic_rows = run_stage(
    "symbolic",
    SYMBOLIC_OUTPUT,
    tuple(
        (
            m
            for m in SELECTED_METHODS
            if m in _api_MistakeBenchmarker.COMPETITOR_SYMBOLIC_METHODS
        )
    ),
)
display(
    MistakeNotebook.comparison_table(
        symbolic_rows, tolerance=PRIMARY_TOLERANCE, rate=INJECTION_RATE
    )
)
print(
    "Attune (no refinement) = weighted string editing on original, unmerged performed MIDI; no repeat correction."
)


Reused 8580 saved rows from /Users/sarah/Desktop/shasha/Attune/benchmarks/results/mistake_competitors_coco_symbolic_laddersym_2026-09-28; no detection or model inference. Results retain their recorded algorithm versions.


,Cases,tp,fp,fn,Precision %,Recall %,F1 %
method,,,,,,,
Attune (no refinement),26,74,2,0,97.4,100.0,98.7
Parangonar DualDTW,26,72,2,2,97.3,97.3,97.3
Parangonar Automatic,26,67,1,7,98.5,90.5,94.4
Nakamura,26,71,21,3,77.2,95.9,85.5
Parangonar TheGlueNote,26,71,31,3,69.6,95.9,80.7


Attune (no refinement) = weighted string editing on original, unmerged performed MIDI; no repeat correction.


## Run comparison

Primary score: pooled missed/extra-event F1 at **100 ms / 50 cents**. Substitutions count as missed + extra, giving equivalent edits equal credit. Short/long errors are saved separately.

Runs CPU preparation and aligners with eight processes first, then releases those processes and runs pending PolyTune and LadderSym cases concurrently through a persistent model pool sized from available RAM and CPU cores at that point. Audio files, compatible predictions, and checkpoints are reused; each pool slot holds one model at a time and reuses it across cases of that method. Workers share read-only memory-mapped checkpoint weights through the OS; each retains its own inference state. LadderSym receives the **clean reference MIDI prompt**, never the injected performance MIDI. It uses the author’s unmodified inference handler and official prompted CocoChorales checkpoint. Its missed/extra outputs are scored directly, without an Attune alignment stage. LadderSym is a general music error detector evaluated here on monophonic stems, not a monophonic-only model. Set `POLYTUNE_LAST = False` to compare with overlapping execution.

In [4]:
importlib.reload(ResultOverview)
overview = MistakeNotebook.overview
rows = run_stage("audio", OUTPUT, SELECTED_METHODS)
report = MistakeNotebook.overview(
    OUTPUT,
    tolerance=PRIMARY_TOLERANCE,
    rate=INJECTION_RATE,
    symbolic_rows=symbolic_rows,
    symbolic_output=SYMBOLIC_OUTPUT,
)
for key, filename in [
    ("leaderboard", "leaderboard.csv"),
    ("clean", "clean_controls.csv"),
    ("extraction", "extraction.csv"),
]:
    temporary = OUTPUT / (filename + ".tmp")
    report[key].to_csv(temporary)
    temporary.replace(OUTPUT / filename)
display(report["leaderboard"])
print(f"{report['injected_cases']} injected cases. Results: {OUTPUT}")
attune_configs = [
    json.loads(value)
    for value in rows.loc[rows.method.str.startswith("Attune"), "config"].unique()
]
if not attune_configs or any(
    (any((c[k] != DEFAULTS[k] for k in DISPLAY_FIELDS)) for c in attune_configs)
):
    raise RuntimeError("Attune result settings do not match the displayed run settings")
print("Verified: Attune rows use the run settings shown above.")


Reused 12948 saved rows from /Users/sarah/Desktop/shasha/Attune/benchmarks/results/mistake_competitors_coco_unassisted_full_2026-09-29; no detection or model inference. Results retain their recorded algorithm versions.


,Precision %,Recall %,F1 %
method,,,
Attune (no refinement),94.7,95.9,95.3
Attune (repeat only),94.7,95.9,95.3
Parangonar DualDTW,92.2,95.9,94.0
Parangonar Automatic,90.4,89.2,89.8
Nakamura,76.9,94.6,84.8
Parangonar TheGlueNote,65.0,90.5,75.7
Attune (Checker 3),72.5,50.0,59.2
LadderSym,23.4,20.3,21.7
PolyTune,26.8,14.9,19.1


26 injected cases. Results: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/mistake_competitors_coco_unassisted_full_2026-09-29
Verified: Attune rows use the run settings shown above.


## Previous run versus new defaults

This compares the same injected sources and seeds against the preserved historical run. It measures the combined change in pitch processing, note segmentation, and alignment, rather than isolating the alignment weight. Repeat-only is the production pipeline; the unrefined row connects to the sweeps.


In [5]:
if (PREVIOUS_OUTPUT / "rows.csv").exists():
    previous = pd.read_csv(PREVIOUS_OUTPUT / "rows.csv")
    keys = ["case_id", "source", "seed", "rate"]

    def injected_sample(frame):
        subset = frame[frame.input.eq("detected") & frame.rate.eq(INJECTION_RATE)]
        return set(map(tuple, subset[keys].drop_duplicates().to_numpy()))

    if injected_sample(previous) == injected_sample(rows):
        old = MistakeNotebook.comparison_table(
            previous,
            input_kind="detected",
            tolerance=PRIMARY_TOLERANCE,
            rate=INJECTION_RATE,
        )
        new = MistakeNotebook.comparison_table(
            rows,
            input_kind="detected",
            tolerance=PRIMARY_TOLERANCE,
            rate=INJECTION_RATE,
        )
        change = old[["tp", "fp", "fn", "F1 %"]].join(
            new[["tp", "fp", "fn", "F1 %"]], lsuffix=" old", rsuffix=" new", how="inner"
        )
        change["F1 gain (pp)"] = change["F1 % new"] - change["F1 % old"]
        display(change)
    else:
        print(
            "Generated cases or samples differ; skipping a paired before/after table."
        )
else:
    print(
        "No previous run available; use the new leaderboard and clean-control table above."
    )


Generated cases or samples differ; skipping a paired before/after table.


## Extraction diagnostic

Detected versus exact performed notes from the paired symbolic run for the same aligner. A large gap points toward pitch/note extraction. PolyTune and LadderSym have no note-input mode; both refinement variants are omitted here because symbolic input uses the original, unmerged notes.

In [6]:
display(report["extraction"])


,Detected notes F1 %,Exact notes F1 %
method,,
Attune (no refinement),95.3,98.7
Nakamura,84.8,85.5
Parangonar Automatic,89.8,94.4
Parangonar DualDTW,94.0,97.3
Parangonar TheGlueNote,75.7,80.7


## Clean false positives

No injected mistakes; fewer false alarms is better. Retain every clean seed evaluation, including differing neural predictions. False alarms and note denominators are pooled over the same evaluations; unique stems and differing-seed counts are displayed separately.

In [7]:
display(report["clean"])


,Clean stems,Clean evaluations,False alarms,False alarms / 100 notes,Stems with differing seed results
method,,,,,
Attune (Checker 3),13,26,0,0.00,0
Attune (no refinement),13,26,0,0.00,0
Attune (repeat only),13,26,0,0.00,0
Nakamura,13,26,0,0.00,0
Parangonar Automatic,13,26,0,0.00,0
Parangonar DualDTW,13,26,0,0.00,0
PolyTune,13,26,8,1.28,0
Parangonar TheGlueNote,13,26,12,1.92,0
LadderSym,13,26,22,3.51,0


## Optional: inspect injected false positives

Replay cached pitches to examine pitch support, segmentation and alignment. Categories are diagnostic evidence, not proven causes. [Historical audit](../modules/mistake/INJECTED_CASE_AUDIT.md).

## CPU timing

Worker-local CPU time excludes queue waits. Aligners include shared pitch/note extraction; PolyTune/LadderSym model setup is separate and charged once per worker. Older checkpoints retain F1 but have no CPU measurement—check coverage.

In [9]:
timing = _api_MistakeBenchmarker.MistakeBenchmarker.summarize(rows)
timing = timing[
    (timing.input != "oracle_notes")
    & (timing.metric == "audio_pitch")
    & (timing.rate == INJECTION_RATE)
    & (timing.tolerance == PRIMARY_TOLERANCE)
]
columns = [
    "method",
    "cases",
    "cpu_timed_cases",
    "mean_cpu_seconds",
    "mean_execution_cpu_seconds",
    "mean_setup_cpu_seconds",
    "mean_wall_seconds",
]
display(timing[[c for c in columns if c in timing]])


,method,cases,cpu_timed_cases,mean_cpu_seconds,mean_execution_cpu_seconds,mean_setup_cpu_seconds,mean_wall_seconds
23,LadderSym,26,26,153.056421,9.964628,1.311445,9.476287
32,PolyTune,26,26,20.213494,0.000000,0.000000,0.000000
280,Attune (Checker 3),26,26,4.386833,0.880276,NaN,1.280163
313,Attune (no refinement),26,26,4.080952,0.574395,NaN,0.888535
346,Attune (repeat only),26,26,4.089054,0.582497,NaN,0.900547
379,Nakamura,26,26,4.103424,0.596867,NaN,0.955945
412,Parangonar Automatic,26,26,4.332790,0.826233,NaN,1.253693
445,Parangonar DualDTW,26,26,4.396777,0.890220,NaN,1.368064
478,Parangonar TheGlueNote,26,26,4.337170,0.830613,NaN,1.438431


## Stage audit on this exact injected set

Stage audits are separate diagnostic replays, not part of Run All. A previous audit may use older pitch, note, or alignment defaults, so it is never displayed automatically here. To regenerate an audit, use the commands in the [README](../modules/mistake/README.md) with `OUTPUT` set to this notebook's current result directory.


In [10]:
print("Optional stage audit output:", OUTPUT / "pipeline_audit")
print(
    "Regenerate PipelineAudit and PipelineAuditDetails for this completed run before interpreting stage-level metrics."
)


Optional stage audit output: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/mistake_competitors_coco_unassisted_full_2026-09-29/pipeline_audit
Regenerate PipelineAudit and PipelineAuditDetails for this completed run before interpreting stage-level metrics.


## Native author dataset: CocoChorales-E

These independent cells compare **Attune, PolyTune, and LadderSym** on the authors' original
paired audio, score MIDI, and extra/missed/correct MIDI labels. **No resynthesis or local
error injection** is performed. PolyTune/LadderSym export three semantic note classes
(1=extra, 2=missed, 3=correct), each with pitch, onset, and offset; our JSON preserves
class identity even when a MIDI track is empty. Substitutions are missed + extra.

Sources: [PolyTune](https://github.com/ben2002chou/Polytune),
[LadderSym](https://github.com/ben2002chou/LadderSYM),
[CocoChorales-E](https://huggingface.co/datasets/ben2002chou/CocoChorales-E).

**Scope:** the pinned Coco mirror is explicitly the authors' **LadderSym subset**;
its equivalence to PolyTune's original Globus release has not been established.
The default is three deterministically selected test **pieces**, all available stems
per Coco piece. Set `NATIVE_MAX_PIECES = None`
for the complete author-mirror test split. Subset scores are not paper reproductions.
The dataset manifests record revisions, official test membership, selection, and file hashes.

Only individual Coco instrument stems are evaluated; ensemble mixes are excluded.

**Attune input policy:** source audio remains unchanged on disk. Attune resamples it
to 44.1 kHz so its 4096-sample window and 128-sample hop retain the earlier benchmark’s
92.9 ms and 2.9 ms durations. Its search range covers every performed pitch (the union
of correct + extra labels, excluding missed notes), with the same ±4-semitone margin
as the earlier injected benchmark. This is **oracle-assisted range selection**, not
a fully blind evaluation; label timings and individual note identities are not supplied
to the detector. Set `NATIVE_RANGE_POLICY="fixed"` for a fixed-range ablation with a new run tag.
Native Attune uses a **0.5-semitone pitch-error threshold** and a **100 ms cap on
the score-derived segmentation minimum** (shorter score-derived minima stay shorter).
Both halves of a substitution are exported at the detected replacement note's time,
matching the native convention; pure deletions retain original score time.
No label timestamps enter inference. Onset-span fitting, robust matched-onset refitting
and production repeat refinement remain enabled. Equal-neighbor errors stay in the dataset.
The audio models use the appropriate official dataset checkpoint (LadderSym prompted),
with contiguous inference enabled as in the upstream evaluation commands.

**Metrics:** `common_pooled_events` reports missed/extra pooled F1 at 50/100/200 ms.
`native_style_macro` reports per-case extra/missed/correct onset F1 and their equal average
at 50 ms / 50 cents, no offsets, using `mir_eval`; empty classes score zero.
This explicit semantic-class evaluator avoids upstream MIDI empty-track renumbering.
It is **not a literal invocation of the upstream evaluator**, nor its Coco instrument-macro
aggregation; retain that distinction when comparing to a paper table.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

REPO_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").exists()
    )
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from benchmarks.modules.mistake.datasets.NativeDatasets import prepare, prepare_models

NATIVE_MAX_PIECES = 3
NATIVE_SEED = 0
NATIVE_METHODS = ("Attune", "PolyTune", "LadderSym")
NATIVE_DEVICE = "cpu"
NATIVE_WORKERS = None
NATIVE_NEURAL_WORKERS = None
NATIVE_RANGE_POLICY = "performed_notes"
NATIVE_MIDI_RANGE = (21, 108)
NATIVE_REUSE_AUDIO = (
    REPO_ROOT / "benchmarks/results/native_v4_no_boundary_assistance_seed0_pieces3"
)
NATIVE_RUN_TAG = "native_v5_min100_seed0_pieces3"
NATIVE_DATA = REPO_ROOT / "benchmarks/datasets/native_errors"
NATIVE_RESULTS = REPO_ROOT / "benchmarks/results" / NATIVE_RUN_TAG


def run_native(dataset, manifest):
    output = NATIVE_RESULTS / dataset
    request = dict(
        manifest_path=str(manifest),
        output=str(output),
        methods=NATIVE_METHODS,
        device=NATIVE_DEVICE,
        midi_range=NATIVE_MIDI_RANGE,
        tolerances=(0.05, 0.1, 0.2),
        workers=NATIVE_WORKERS,
        neural_workers=NATIVE_NEURAL_WORKERS,
        range_policy=NATIVE_RANGE_POLICY,
        reuse_audio_from=(
            str(NATIVE_REUSE_AUDIO / dataset) if NATIVE_REUSE_AUDIO else None
        ),
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "benchmarks.modules.mistake.MistakeBenchmarker",
            "native",
        ],
        input=json.dumps(request),
        text=True,
        cwd=REPO_ROOT,
        check=True,
    )
    summary = pd.read_csv(output / "summary.csv")
    print(
        "Raw predictions, labels, resumable checkpoints, logs and provenance:", output
    )
    return summary


### Download the selected Coco cases and verify checkpoints

Running this cell downloads only selected test inputs/labels, plus missing official weights.
Existing files are reused and model SHA-256 hashes are verified. Both model environments
must already be installed using the setup commands above. No models run in this cell.

In [2]:
native_manifest = prepare(
    "coco", NATIVE_DATA / "coco", max_pieces=NATIVE_MAX_PIECES, seed=NATIVE_SEED
)
prepare_models("coco", methods=NATIVE_METHODS)
selection = json.loads(native_manifest.read_text())
print(
    "coco",
    selection["scope"],
    "—",
    len(selection["selected_pieces"]),
    "pieces,",
    len(selection["cases"]),
    "cases; manifest:",
    native_manifest,
)


coco: preparing piece 1/3: 239244
coco: preparing piece 2/3: 224962
coco: preparing piece 3/3: 224806
coco: verifying/downloading official PolyTune weights: /Users/sarah/Desktop/shasha/Attune/benchmarks/datasets/pretrained/polytune/coco.ckpt
coco: verifying/downloading official LadderSym weights: /Users/sarah/Desktop/shasha/Attune/benchmarks/datasets/pretrained/laddersym/coco_prompted.ckpt
coco test subset — 3 pieces, 12 cases; manifest: /Users/sarah/Desktop/shasha/Attune/benchmarks/datasets/native_errors/coco/manifest_seed0_pieces3.json


### CocoChorales-E: authored instrument stems

Run all three systems on the same selected official test stems. Results are written
separately from every synthetic benchmark above. Each completed case/method is checkpointed;
rerunning resumes matching inputs/settings/code. Changed inference provenance requires a new run tag.

Attune cases run in parallel processes. After those exit, a RAM-bounded pool runs all
PolyTune cases, then all LadderSym cases, retaining one loaded model per worker.
`NATIVE_WORKERS` and `NATIVE_NEURAL_WORKERS` override automatic sizing.
The same compact progress renderer as the earlier cells shows **completed/total, method,
and stem/input**, plus worker counts when each stage starts; for example:
`[  17/36] PolyTune: 239244/0_flute audio`. The total is **stems × selected methods**,
including cached results. All updates overwrite one line. Worker output goes to case/model logs.
`NATIVE_REUSE_AUDIO` points to the preserved prior run. Compatible PolyTune/LadderSym
predictions are reused and rescored before fresh Attune work; missing cases run normally.
Set it to `None` to disable cross-run reuse.

Native Attune now stores pitch and unrefined note stages separately. Changes to the
error threshold reuse both; changes to the segment minimum reuse pitches and redo notes.
Completed matching cases still skip all inference. Earlier native runs saved final
events but no stage caches; those missing stages can only be populated on a future
user-triggered run. Cache-only updates preserve the completed v4 results.


In [13]:
native_coco_summary = run_native("coco", native_manifest)


[  36/36] Attune: 224806/1_french_horn detected                 
Raw predictions, labels, resumable checkpoints, logs and provenance: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/native_v5_min100_seed0_pieces3/coco


In [14]:
from pathlib import Path
import sys
from IPython.display import display, Markdown

_native_root = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").exists()
    )
)
if str(_native_root) not in sys.path:
    sys.path.insert(0, str(_native_root))
from benchmarks.modules.mistake.MistakeNotebook import MistakeNotebook

_native_output = (
    globals().get(
        "NATIVE_RESULTS",
        _native_root / "benchmarks/results/native_v5_min100_seed0_pieces3",
    )
    / "coco"
)
display(Markdown("### Native Coco-E — F1 summary across all cases"))
if (_native_output / "rows.csv").exists():
    native_f1_summary = MistakeNotebook.native_overview(_native_output)
    display(native_f1_summary)
else:
    print("No saved native results yet. Run the comparison cell above first.")


### Native Coco-E — F1 summary across all cases

,Method,Cases,"Mean case error F1 (100 ms), %","Pooled error F1 (100 ms), %","Mean three-class F1 (50 ms), %",TP,FP,FN
0,Attune,12/12,52.5,60.4,55.4,16,15,6
1,PolyTune,12/12,75.6,85.7,66.9,21,6,1
2,LadderSym,12/12,79.4,80.0,67.4,18,5,4


### How to read this table

All F1 columns are **percentages**: `3.2` means 3.2%, not 0.032%.

- **Mean case error F1 (100 ms):** calculate missed+extra F1 for each stem and average all
  stems equally. For this reporting column only, a clean case with no predicted errors counts
  as 100%; a clean case with false alarms counts as 0%. Raw undefined F1 values remain unchanged.
- **Pooled error F1 (100 ms):** add TP, FP and FN over every stem, then calculate
  `2 TP / (2 TP + FP + FN)`. This is the same *kind of metric* as the earlier synthetic
  benchmark's headline F1, but uses a different dataset and input policy.
- **Mean three-class F1 (50 ms):** average extra, missed, and correct-note F1 equally,
  then average stems. It has a stricter timing gate and includes correct notes. Empty
  classes retain the upstream-style zero convention: a perfectly transcribed clean stem
  with no missed/extra labels can have only 33.3% in this column. Do not compare this
  directly with the pooled error F1 column.
- **TP / FP / FN** belong to the pooled 100 ms error metric. Cases shows coverage;
  aggregate F1 is left blank until every selected case for that method is available.

**Configured next run:** `native_v5_min100_seed0_pieces3` uses the **100 ms segmentation cap**,
keeping native timestamps and the 0.5-semitone pitch threshold. This setting has not
been benchmarked. Run All will reuse compatible upstream caches and neural predictions.

**Historical 50 ms run:** `native_v4_no_boundary_assistance_seed0_pieces3` adds native
substitution timestamps, a 0.5-semitone pitch threshold and the 50 ms segmentation cap.
Attune now has **37.7% mean case F1**, **49.2% pooled F1**, and **16 TP / 27 FP / 6 FN**.
The combined changes recover five more labeled events but also add fifteen false positives;
they do not close the gap. These are jointly changed settings, not isolated causal ablations.
PolyTune/LadderSym predictions are verified and reused; their scores remain unchanged.

**Alignment versus boundary correction:** Coco-E already uses audio-based onset fitting,
robust matched-onset refitting and repeat refinement. The historical injected benchmark used known performed-MIDI durations to adjust the
first/last detected notes. That oracle correction has now been removed from injected
comparisons and audits too; old trimmed or unmarked note caches are regenerated. Substitution
export now uses the detected replacement's onset/end; no truth timing is supplied.
A shorter segment minimum can help short notes, but cannot guarantee separation of
adjacent equal-pitch notes: the detector also consolidates segments of similar pitch.

**Previous run:** `native_v2_44100_performed_range_seed0_pieces3` uses Attune audio
resampled to 44.1 kHz and an oracle-assisted performed-note pitch range (±4 semitones).
Across all 12 stems, Attune has **45.2% mean case error F1**, **48.9% pooled error F1**,
and 11 TP / 12 FP / 11 FN at 100 ms. PolyTune remains 85.7% pooled and LadderSym 80.0%.
Their predictions were reused only after checking case hashes, model identities,
packages and adapter code, then rescored. Their input audio remains at the author rate.

**Historical diagnosis:** the original `native_v1_seed0_pieces3` run is preserved.
Its Attune `0.031746` was extra-note-only pooled F1 at 50 ms (1 TP / 51 FP / 10 FN),
not overall F1. Combined 100 ms error F1 was 18.4% (8 TP / 57 FP / 14 FN).
The flute had 24 false-positive errors, including many 64–88 ms detected fragments
near pitch transitions. These are detector artifacts, not authored error categories.
The viola also had spurious MIDI 21.6–22.5 detections in a gap/tail under the full-piano range.
The previous v2 run has zero flute false positives, but still misses one labeled event.

**Why the sample rate mattered:** the downloaded WAVs really are 16 kHz; the adapter
had not downsampled them. Keeping 4096/128 samples at that rate accidentally changed
Attune's window/hop to 256/8 ms instead of 92.9/2.9 ms. A window spanning a pitch
transition can yield unstable/intermediate pitches, which segmentation may split into
short notes. The longer window is a plausible contributor to these false alarms.
Resampling fixes the time grid; it cannot restore frequencies absent from the 16 kHz audio.
Both frontend changes were applied together, so the new aggregate measures their combined effect.
An earlier isolated flute resampling test (range unchanged) reduced false positives
from 24 to 4; artifacts are in `benchmarks/results/native_reporting_audit_2026-09-29/`.

**Why this differs from our injected CocoChorales:** both are synthetic, but our earlier
benchmark injects local monophonic edits and renders them with FluidSynth/MuseScore at
44.1 kHz. CocoChorales-E supplies the authors' own error edits, labels and MIDI-DDSP
renderings (16 kHz in this mirror), with different timbres, transitions, error durations,
pieces and error density. The author algorithm includes omissions, pitch changes,
time shifts and added notes ([paper](https://arxiv.org/html/2501.02030v1)).
A native flute substitution here lasts only 55 ms. The previous 95.3% synthetic
result is not a before/after score on these recordings, nor held-out validation.

Detailed class/gate rows remain available in `native_coco_summary` and `summary.csv`.
The summary cell above only reads saved files and can be rerun without model inference.


## Controlled rendering experiment: native COCO-E with our soundfont

Run these cells manually. Use the same 12 native stems, original score/performance MIDI,
and byte-identical labels as the completed v5 run, replacing **both** audio inputs with
FluidSynth/MuseScore renders. Original performed MIDI is downloaded from the pinned
public dataset revision and checked against correct+extra labels, including note
pitch, timing, program and velocity. No local mistakes or truth boundary corrections
are introduced. Synthesized audio is converted to native 16 kHz mono PCM24.

Preparation renders stems in parallel. Inference uses the **same automatically sized
parallel CPU workers and RAM-bounded persistent neural-worker pools as the native cells**:
Attune first, then PolyTune, then LadderSym. Progress overwrites one line using the same
renderer; worker details remain in logs. All outputs go into a separate resumable folder.
New audio requires fresh inference: original neural predictions are never reused.

This isolates replacement by our rendering pipeline, including its attack/release,
tuning and loudness characteristics; it does not isolate timbre alone. Existing native
results and production settings are preserved. No benchmark inference has been run for
this experiment. A stopped preparation attempt may have left intermediate files, which
are safely re-rendered until a complete manifest exists.


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display

SF_REPO = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").exists()
    )
)
SF_BASE = SF_REPO / "benchmarks/results/native_v5_min100_seed0_pieces3/coco"
SF_OUTPUT = SF_REPO / "benchmarks/results/native_soundfont_2026-09-29"
SF_RENDER_WORKERS = None
SF_WORKERS = None
SF_NEURAL_WORKERS = None


def soundfont_command(action):
    cmd = [
        sys.executable,
        "-m",
        "benchmarks.modules.mistake.MistakeBenchmarker",
        "soundfont",
        action,
        "--base",
        str(SF_BASE),
        "--output",
        str(SF_OUTPUT),
    ]
    for flag, value in (
        ("--render-workers", SF_RENDER_WORKERS),
        ("--workers", SF_WORKERS),
        ("--neural-workers", SF_NEURAL_WORKERS),
    ):
        if value is not None:
            cmd.extend([flag, str(value)])
    return cmd


In [ ]:
subprocess.run(soundfont_command("prepare"), cwd=SF_REPO, check=True)


[  12/12] render: 224806/2_trombone           


CompletedProcess(args=['/Users/sarah/Desktop/shasha/Attune/at-venv/bin/python', '-m', 'benchmarks.experiments.native_soundfont_2026_09_29.run', 'prepare', '--base', '/Users/sarah/Desktop/shasha/Attune/benchmarks/results/native_v5_min100_seed0_pieces3/coco', '--output', '/Users/sarah/Desktop/shasha/Attune/benchmarks/results/native_soundfont_2026-09-29'], returncode=0)

In [5]:
subprocess.run(soundfont_command("run"), cwd=SF_REPO, check=True)


[  12/36] PolyTune: 2 persistent model workers                  

/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/transcription.py:161: UserWarning: Reference notes are empty.
  warnings.warn("Reference notes are empty.")
/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/transcription.py:163: UserWarning: Estimated notes are empty.
  warnings.warn("Estimated notes are empty.")


[  13/36] PolyTune: 239244/1_oboe audio                         

/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/transcription.py:163: UserWarning: Estimated notes are empty.
  warnings.warn("Estimated notes are empty.")


[  14/36] PolyTune: 239244/0_flute audio                        

...

/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/transcription.py:161: UserWarning: Reference notes are empty.
  warnings.warn("Reference notes are empty.")
/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/transcription.py:163: UserWarning: Estimated notes are empty.
  warnings.warn("Estimated notes are empty.")


[  35/36] LadderSym: 224806/2_trombone audio                    

/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/transcription.py:161: UserWarning: Reference notes are empty.
  warnings.warn("Reference notes are empty.")
/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/transcription.py:163: UserWarning: Estimated notes are empty.
  warnings.warn("Estimated notes are empty.")


[  36/36] LadderSym: 224806/3_tuba audio                        


CompletedProcess(args=['/Users/sarah/Desktop/shasha/Attune/at-venv/bin/python', '-m', 'benchmarks.experiments.native_soundfont_2026_09_29.run', 'run', '--base', '/Users/sarah/Desktop/shasha/Attune/benchmarks/results/native_v5_min100_seed0_pieces3/coco', '--output', '/Users/sarah/Desktop/shasha/Attune/benchmarks/results/native_soundfont_2026-09-29'], returncode=0)

In [6]:
if (SF_OUTPUT / "summary.csv").exists():

    def pooled_100(path):
        table = pd.read_csv(path / "summary.csv")
        return table[
            (table.protocol == "common_pooled_events")
            & (table.metric == "audio_pitch")
            & (table.tolerance == 0.1)
        ].set_index("method")[["tp", "fp", "fn", "f1"]]

    original = pooled_100(SF_BASE)
    rendered = pooled_100(SF_OUTPUT)
    comparison = original.add_prefix("Native ").join(rendered.add_prefix("Soundfont "))
    rows = pd.read_csv(SF_OUTPUT / "rows.csv")
    coverage = (
        rows[
            (rows.protocol == "common_pooled_events")
            & (rows.metric == "audio_pitch")
            & (rows.tolerance == 0.1)
        ]
        .groupby("method")
        .case_id.nunique()
    )
    import json

    expected = len(json.loads((SF_OUTPUT / "manifest.json").read_text())["cases"])
    comparison["Cases"] = coverage.reindex(comparison.index).fillna(0).astype(int)
    comparison.loc[
        comparison["Cases"] != expected,
        ["Soundfont tp", "Soundfont fp", "Soundfont fn", "Soundfont f1"],
    ] = float("nan")
    comparison["Native F1 %"] = 100 * comparison.pop("Native f1")
    comparison["Soundfont F1 %"] = 100 * comparison.pop("Soundfont f1")
    comparison["Change (pp)"] = comparison["Soundfont F1 %"] - comparison["Native F1 %"]
    display(comparison.round(2))
else:
    print("No soundfont results yet. Run preparation and inference above.")


,Native tp,Native fp,Native fn,Soundfont tp,Soundfont fp,Soundfont fn,Cases,Native F1 %,Soundfont F1 %,Change (pp)
method,,,,,,,,,,
Attune,16.0,15.0,6.0,16.0,4.0,6.0,12,60.38,76.19,15.81
LadderSym,18.0,5.0,4.0,7.0,16.0,15.0,12,80.00,31.11,-48.89
PolyTune,21.0,6.0,1.0,4.0,18.0,18.0,12,85.71,18.18,-67.53


## COCO-E: cached Attune pitch and note diagnostics

Read-only: no synthesis, inference, fitting, or cache updates. Compares the native v5
run with the soundfont run. `correct + extra` MIDI labels describe all performed notes.
The native download contains no acoustic F0 reference for these error performances,
so **MIDI pitch agreement is a proxy, not acoustic F0 accuracy**. Frames with multiple
simultaneous labeled notes are excluded from the pitch denominator. Interiors exclude
50 ms on both sides of every MIDI boundary. Voicing in MIDI silence includes acoustic
release tails and must not be interpreted automatically as a detector false alarm.
Frame statistics cover 0 to the final performed MIDI offset, equally for both renders.

Note metrics match performed notes one-to-one within 50 cents and 50/100/200 ms onset
windows. `offsets=True` additionally requires offset error ≤ max(50 ms, 20% of reference
duration). These measure extraction, regardless of whether the note was a correct or
mistaken performance. Initial notes come from the saved unrefined cache; final notes
come from the saved correct+extra predictions. Nothing is aligned to labels for scoring.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

STAGE_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").exists()
    )
)
if str(STAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(STAGE_ROOT))
from benchmarks.archive.experiments.native_v5_gap_2026_09_29.stage_accuracy import (
    inspect as inspect_stages,
)

stage_pitch_tables, stage_note_tables, stage_details = ([], [], {})
for label, folder in {
    "Original": "native_v5_min100_seed0_pieces3/coco",
    "FluidSynth": "native_soundfont_2026-09-29",
}.items():
    pitch_table, note_table, details = inspect_stages(
        STAGE_ROOT / "benchmarks/results" / folder
    )
    stage_pitch_tables.append(pitch_table.assign(Renderer=label))
    stage_note_tables.append(note_table.assign(Renderer=label))
    stage_details[label] = details
print("Saved pitch agreement (MIDI proxy) and performed-note extraction accuracy:")
display(pd.concat(stage_pitch_tables, ignore_index=True))
display(pd.concat(stage_note_tables, ignore_index=True))


## Renderer timing confound: cached diagnostic only

The upstream MIDI-DDSP duration sequence serializes overlapping notes. Two native
violin stems appear to follow this delayed schedule rather than their MIDI labels.
This accounts for 27 of 33 unmatched notes at 100 ms in the saved original run.
The hypothesis row is **not corrected ground truth or an official benchmark**:
the exact historical synthesis revision is unavailable. Previous pitch agreement
and note metrics against official MIDI inherit this timing confound.

This cell only reads saved events and MIDI; no synthesis/inference or result writes.
See `docs/native-coco-renderer-timing-2026-09-29.md` for evidence and synthesis options.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

TIMELINE_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").exists()
    )
)
if str(TIMELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(TIMELINE_ROOT))
from benchmarks.modules.mistake.MistakeNotebook import MistakeNotebook

timeline_summary, timeline_details, timeline_boundaries = (
    MistakeNotebook.renderer_inspect(
        TIMELINE_ROOT / "benchmarks/results/native_v5_min100_seed0_pieces3/coco",
        TIMELINE_ROOT / "benchmarks/results/native_soundfont_2026-09-29",
    )
)
display(timeline_summary)
display(
    timeline_details[
        timeline_details["case"].str.contains("violin") & (timeline_details.gate == 0.1)
    ]
)


## Our injected mistakes, rendered with MIDI-DDSP — manual experiment

Uses the **existing MistakeInjector and injected comparison pipeline** on the same
13 source stems. Default constraints are **pitch difference > 1 semitone** for
substitutions and each insertion's final neighbors, and **all performed-note durations
> 0.100 s**. Change `DDSP_MIN_NOTE_SECONDS` for x. Deterministic rejection sampling
checks the final serialized notes before synthesis, records attempts/seeds, and fails
rather than silently weakening constraints. Clean controls use one seed; injected
cases use seeds 0 and 1, with the existing 25% rate and five edit types.

Both **MIDI-DDSP and FluidSynth** render the exact same accepted MIDI and clean scores.
Use this paired control, rather than the previous injected leaderboard, when changing
constraints. Attune uses its injected-benchmark defaults and original-score timestamps
for missed notes. Both renderers are supplied at 44.1 kHz to keep Attune's analysis grid
fixed (MIDI-DDSP originals at 16 kHz are preserved). There is no truth boundary assistance.

**Environment prerequisite:** configure `DDSP_PYTHON` to a separate working MIDI-DDSP
Python environment with its pretrained weights. The [upstream instructions](https://github.com/magenta/midi-ddsp)
use Python 3.8/TensorFlow 2.7 and warn that the package does not install on M1 Macs.
A supported Linux environment is the recommended place to run synthesis. No packages
are installed here, and no FluidSynth fallback is allowed for MIDI-DDSP. In that
separate environment, the documented setup commands are `pip install midi-ddsp` and
`midi_ddsp_download_model_weights`; these are instructions, not executed cells.

This uses the original synthesizer **family**, regenerating expressions with seeded
pretrained MIDI-DDSP. It is not a recreation of archived CocoChorales expression
parameters. Engine/package/weight hashes and generated F0 controls are saved. Rendering
uses multiple persistent processes (load weights once per batch), sized by available RAM;
Attune and neural comparisons retain the existing parallel pools and one-line progress.
Workers log details to files. All results are isolated from completed benchmarks.

Nothing in this section has been executed. `action="prepare"` exports only injected
MIDI, `"render"` renders it, `"evaluate"` compares saved audio, and `"all"` runs all stages.
A missing MIDI-DDSP environment fails before an `"all"` run begins.


In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess

DDSP_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").exists()
    )
)
DDSP_PYTHON = Path(
    os.environ.get("MIDI_DDSP_PYTHON", "/path/to/midi-ddsp-env/bin/python")
)
DDSP_MIN_PITCH_DIFFERENCE = 1.0
DDSP_MIN_NOTE_SECONDS = 0.1
DDSP_OUTPUT = (
    DDSP_ROOT / "benchmarks/results/injected_mididdsp_gap_gt1_min_gt100ms_2026-09-29"
)
DDSP_REQUEST = dict(
    source_run=str(
        DDSP_ROOT
        / "benchmarks/results/mistake_competitors_coco_unassisted_full_2026-09-29/run.json"
    ),
    output=str(DDSP_OUTPUT),
    ddsp_python=str(DDSP_PYTHON),
    min_pitch_difference=DDSP_MIN_PITCH_DIFFERENCE,
    min_seconds=DDSP_MIN_NOTE_SECONDS,
    seeds=[0, 1],
    workers=None,
    neural_workers=None,
    render_workers=None,
    action="all",
)


In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "benchmarks.archive.experiments.injected_mididdsp_2026_09_29.run",
    ],
    input=json.dumps(DDSP_REQUEST),
    text=True,
    cwd=DDSP_ROOT,
    check=True,
)


In [ ]:
import pandas as pd
from IPython.display import display

paired = []
for renderer in ("fluidsynth", "mididdsp"):
    path = DDSP_OUTPUT / renderer / "summary.csv"
    if path.exists():
        table = pd.read_csv(path)
        table = table[(table.metric == "audio_pitch") & (table.tolerance == 0.1)].copy()
        table["Renderer"] = renderer
        table["F1 %"] = 100 * table.f1
        paired.append(
            table[["Renderer", "method", "rate", "cases", "tp", "fp", "fn", "F1 %"]]
        )
if paired:
    display(pd.concat(paired, ignore_index=True).round(2))
else:
    print("No paired MIDI-DDSP injection results yet.")


## URMP repeat cases: before vs collapsed-score alignment (2026-09-30)

Paired comparison on **10 distinct cached URMP recordings containing contiguous score repeats** from the existing repeat corpus. Both methods receive identical saved, pre-refinement audio-derived notes and the official score MIDI. This isolates the alignment/refinement change; it does not rerun the current pitch/note front end. The cohort was selected for matching score/annotation pitch sequences and available caches, so it is development evidence, not a representative or held-out URMP result.

**Before:** original-score alignment, one robust matched-onset fit, and the frozen previous repeat refiner with full-sequence cost checks. **After:** collapsed-score alignment against original detected notes, one robust fit, then local expansion with at most 100 ms additional summed timing distance per region. Existing detected boundaries are preserved. No annotation timing enters either method.

The metric is **note F1**, not mistake-event F1: one-to-one onset/pitch matches at 100 ms / 50 cents, without an offset criterion. These recordings do not supply injected mistake-event truth. The table also reports 50/200 ms sensitivity. “Repeat regions” uses fixed performed-annotation windows around all score repeat groups (first onset −100 ms through last offset +100 ms); **all** reference and estimated onsets in these windows count, including adjacent notes, to avoid treating legitimate neighbors as false positives.

The saved run shows a small regression: whole-recording F1 **92.88% → 92.27%**; repeat-region F1 **88.23% → 86.77%** at 100 ms. No parameters were tuned to improve these results. Historical outputs are preserved. Set `RERUN_REPEAT_COMPARISON = True` to replay into a fresh timestamped folder.


In [1]:
from pathlib import Path
import json
import subprocess
import sys
from datetime import datetime
import pandas as pd
from IPython.display import display

repeat_root = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").is_file()
    )
)
repeat_source = repeat_root / "benchmarks/results/repeat_refinement_urmp_production"
repeat_output = (
    repeat_root / "benchmarks/results/repeat_collapsed_before_after_2026-09-30_final"
)
RERUN_REPEAT_COMPARISON = False
if RERUN_REPEAT_COMPARISON or not (repeat_output / "summary.csv").exists():
    repeat_output = repeat_root / (
        "benchmarks/results/repeat_collapsed_before_after_"
        + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "benchmarks.modules.mistake.sweeps.CollapsedRepeatComparison",
            "--source",
            str(repeat_source),
            "--output",
            str(repeat_output),
        ],
        cwd=repeat_root,
        check=True,
    )
repeat_metadata = json.loads((repeat_output / "run.json").read_text())
assert repeat_metadata["status"] == "complete"
repeat_summary = pd.read_csv(repeat_output / "summary.csv")
repeat_f1 = repeat_summary.pivot(
    index=["scope", "tolerance"], columns="method", values="f1"
)
repeat_f1 = repeat_f1[["Before", "After"]]
repeat_f1["Change (percentage points)"] = repeat_f1["After"] - repeat_f1["Before"]
print(f"{len(repeat_metadata['tracks'])} paired URMP repeat cases; note F1 (%)")
display(repeat_f1.round(3))
display(
    repeat_summary[repeat_summary.tolerance.eq(0.1)][
        ["scope", "method", "tp", "fp", "fn", "precision", "recall", "f1"]
    ].round(3)
)
print(
    f"Saved inputs, code hashes, per-track notes, proposals, and counts: {repeat_output}"
)


10 paired URMP repeat cases; note F1 (%)
Saved inputs, code hashes, per-track notes, proposals, and counts: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/repeat_collapsed_before_after_2026-09-30_final


method                                  Before  ...  Change (percentage points)
scope                        tolerance          ...                            
Repeat regions               0.05       80.929  ...                      -2.219
                             0.10       88.226  ...                      -1.451
                             0.20       91.542  ...                      -0.897
Whole repeat-case recordings 0.05       87.640  ...                      -0.813
                             0.10       92.884  ...                      -0.614
                             0.20       94.632  ...                      -0.383

[6 rows x 3 columns]

,scope,method,tp,fp,fn,precision,recall,f1
2,Repeat regions,After,269,51,31,84.062,89.667,86.774
3,Repeat regions,Before,266,37,34,87.789,88.667,88.226
8,Whole repeat-case recordings,After,746,75,50,90.865,93.719,92.270
9,Whole repeat-case recordings,Before,744,62,52,92.308,93.467,92.884


## Follow-up: preserve already detected repeats across release gaps

The first collapsed version introduced 14 new false-positive note onsets while removing one old false positive (net +13). Seven of those introduced false positives disappear when adjacent same-pitch detections are retained across release gaps; see `repeat_gap_aware_2026-09-30/false_positive_audit.json` for every case. A clarinet example already had two A notes separated by 424 ms, but the 50 ms collection limit excluded the second and caused the first to be split again.

The correction removes that performed-gap ceiling. Score contiguity is unchanged. The coarse alignment still bounds each local region, and a different pitch or another score correspondence stops collection. No global rerun or new threshold is added. The duration-only cost gate can still accept unsupported proportional splits, so improvement here is not evidence that every inferred attack is correct.

The table preserves all three versions on identical saved detections. This targeted correction was evaluated on the same 10-track development cohort. Set `RERUN_GAP_COMPARISON = True` for a fresh replay into a new output directory.


In [1]:
from pathlib import Path
from datetime import datetime
import json
import sys
import subprocess
import pandas as pd
from IPython.display import display

gap_root = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").is_file()
    )
)
gap_output = gap_root / "benchmarks/results/repeat_gap_aware_2026-09-30"
RERUN_GAP_COMPARISON = False
if RERUN_GAP_COMPARISON or not (gap_output / "summary.csv").exists():
    gap_output = gap_root / (
        "benchmarks/results/repeat_gap_aware_"
        + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "benchmarks.modules.mistake.sweeps.CollapsedRepeatComparison",
            "--source",
            str(gap_root / "benchmarks/results/repeat_refinement_urmp_production"),
            "--output",
            str(gap_output),
        ],
        cwd=gap_root,
        check=True,
    )
assert json.loads((gap_output / "run.json").read_text())["status"] == "complete"
gap_summary = pd.read_csv(gap_output / "summary.csv")
first_summary = pd.read_csv(
    gap_root
    / "benchmarks/results/repeat_collapsed_before_after_2026-09-30_final/summary.csv"
)
gap_table = gap_summary.pivot(
    index=["scope", "tolerance"], columns="method", values="f1"
)
gap_table = gap_table.rename(
    columns={"Before": "Old flow", "After": "Gap-aware collapsed"}
)
gap_table["First collapsed"] = (
    first_summary[first_summary.method.eq("After")].set_index(["scope", "tolerance"]).f1
)
gap_table = gap_table[["Old flow", "First collapsed", "Gap-aware collapsed"]]
gap_table["Change vs old (pp)"] = (
    gap_table["Gap-aware collapsed"] - gap_table["Old flow"]
)
display(gap_table.round(3))
display(
    gap_summary[gap_summary.tolerance.eq(0.1)][
        ["scope", "method", "tp", "fp", "fn", "f1"]
    ].round(3)
)


method                                  Old flow  ...  Change vs old (pp)
scope                        tolerance            ...                    
Repeat regions               0.05         80.929  ...               1.444
                             0.10         88.226  ...               2.384
                             0.20         91.542  ...               3.021
Whole repeat-case recordings 0.05         87.640  ...               0.639
                             0.10         92.884  ...               0.882
                             0.20         94.632  ...               1.129

[6 rows x 4 columns]
                          scope  method   tp  fp  fn      f1
2                Repeat regions   After  275  32  25  90.610
3                Repeat regions  Before  266  37  34  88.226
8  Whole repeat-case recordings   After  752  56  44  93.766
9  Whole repeat-case recordings  Before  744  62  52  92.884


## Original-note local DP with deletion-neutral recovery fees (2026-10-01)

Production now aligns original notes, fits the score once, then considers partial or complete recovery of deletions inside same-pitch repeat groups. Existing matches remain fixed. A split pays the exact original deletion cost of every recovered score note. Unrecovered notes keep their deletion cost; equal-cost choices prefer fewer boundaries. Split times follow the assigned score-duration proportions, with each new piece longer than 30 ms. No acoustic evidence or across-silence note collection is used.

The local DP can leave score notes missing before, between, or after assignments. For K score notes and M matched user notes there are at most K−M additional boundaries and K−M+1 pieces for one user note, further limited by neighboring original matches. Insertions and pitch-incompatible matches bound local blocks. The former aggregate timing-slack gate is no longer used.

This exact fee greatly reduces recovery on the same 10-track development cohort. At 100 ms / 50 cents, whole-recording F1 is **88.86%** (666 TP / 37 FP / 130 FN), versus **93.38%** without fees and **92.88%** for the old global-cost flow. Repeat-region F1 is **75.15%**, versus **89.40%** without fees. Precision improves, but recall drops. Historical results are preserved; parameters were not adjusted to hide the regression. Both versions use identical saved audio-derived notes, not fresh pitch inference.


In [1]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display

fee_root = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").is_file()
    )
)
fee_output = fee_root / "benchmarks/results/repeat_deletion_fee_2026-10-01"
RERUN_FEE_COMPARISON = False
if RERUN_FEE_COMPARISON or not (fee_output / "summary.csv").exists():
    fee_output = fee_root / (
        "benchmarks/results/repeat_deletion_fee_"
        + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "benchmarks.modules.mistake.sweeps.CollapsedRepeatComparison",
            "--source",
            str(fee_root / "benchmarks/results/repeat_refinement_urmp_production"),
            "--output",
            str(fee_output),
        ],
        cwd=fee_root,
        check=True,
    )
assert json.loads((fee_output / "run.json").read_text())["status"] == "complete"
fee_summary = pd.read_csv(fee_output / "summary.csv")
fee_table = fee_summary.pivot(
    index=["scope", "tolerance"], columns="method", values="f1"
)
fee_table = fee_table.rename(
    columns={"Before": "Old flow", "After": "Local DP + recovery fee"}
)
for label, folder in [
    ("Local recovery without fee", "repeat_original_local_2026-10-01_verified"),
    ("Gap-aware collapsed", "repeat_gap_aware_2026-09-30"),
]:
    historical = pd.read_csv(fee_root / "benchmarks/results" / folder / "summary.csv")
    fee_table[label] = (
        historical[historical.method.eq("After")].set_index(["scope", "tolerance"]).f1
    )
display(
    fee_table[
        [
            "Old flow",
            "Gap-aware collapsed",
            "Local recovery without fee",
            "Local DP + recovery fee",
        ]
    ].round(3)
)
display(
    fee_summary[fee_summary.tolerance.eq(0.1)][
        ["scope", "method", "tp", "fp", "fn", "precision", "recall", "f1"]
    ].round(3)
)


## Duration-only local recovery with the same deletion-neutral fee

Benchmark-only experiment: initial pitch/time alignment and robust fitting remain unchanged. Local recovery scores **both unchanged and split alternatives using only absolute duration error**, plus unchanged deletion costs and exact recovery fees. No pitch or onset error enters the local objective. Pitch eligibility (0.5 semitones), original matches, minimum piece duration, and fewer-cuts tie-breaking remain unchanged. Production has not been switched to this experiment.

At 100 ms / 50 cents on the same 10 saved URMP cases, whole-recording F1 improves from **88.86% to 94.03%**, and repeat-region F1 from **75.15% to 91.09%**, relative to pitch-cost-plus-fee recovery. Whole-recording counts are **748 TP / 47 FP / 48 FN**. This isolates removal of the local pitch cost under the current duration-only timing defaults; no acoustic evidence or new threshold was added. These are development-cohort results, not held-out validation.


In [1]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys
import json
import pandas as pd
from IPython.display import display

duration_root = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").is_file()
    )
)
duration_output = (
    duration_root / "benchmarks/results/repeat_duration_only_fee_2026-10-01"
)
RERUN_DURATION_COMPARISON = False
if RERUN_DURATION_COMPARISON or not (duration_output / "summary.csv").exists():
    duration_output = duration_root / (
        "benchmarks/results/repeat_duration_only_fee_"
        + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "benchmarks.modules.mistake.sweeps.DurationOnlyRepeatComparison",
            "--source",
            str(duration_root / "benchmarks/results/repeat_refinement_urmp_production"),
            "--output",
            str(duration_output),
        ],
        cwd=duration_root,
        check=True,
    )
assert json.loads((duration_output / "run.json").read_text())["status"] == "complete"
duration_summary = pd.read_csv(duration_output / "summary.csv")
duration_table = (
    duration_summary[duration_summary.method.eq("After")]
    .set_index(["scope", "tolerance"])[["f1"]]
    .rename(columns={"f1": "Duration only + fee"})
)
for label, folder in [
    ("Pitch + duration + fee", "repeat_deletion_fee_2026-10-01"),
    ("Local recovery without fee", "repeat_original_local_2026-10-01_verified"),
    ("Gap-aware collapsed", "repeat_gap_aware_2026-09-30"),
]:
    previous = pd.read_csv(
        duration_root / "benchmarks/results" / folder / "summary.csv"
    )
    duration_table[label] = (
        previous[previous.method.eq("After")].set_index(["scope", "tolerance"]).f1
    )
display(duration_table.round(3))
display(
    duration_summary[duration_summary.tolerance.eq(0.1)][
        ["scope", "method", "tp", "fp", "fn", "precision", "recall", "f1"]
    ].round(3)
)


## Production promotion — 2026-10-01

Duration-only repeat recovery with deletion-neutral fees is now the production default (cache version 29). The direct production replay in `benchmarks/results/repeat_duration_production_2026-10-01/` exactly reproduces every per-track count and detected note from the duration-only experiment above: **94.03%** whole-recording note F1 and **91.09%** repeat-region F1 at 100 ms / 50 cents. Initial alignment remains pitch/time-aware. **66 focused tests pass.** Restart the app and click Analyze for existing takes; compatible pitch caches are retained.


## Shared pitch tolerance for repeat eligibility — production default

The fixed 0.5-semitone gate is replaced by the same classification helper used for pitch-mistake reporting: **distance < configured pitch tolerance** is eligible; **distance ≥ tolerance** is a substitution and splits the region. Initial alignment, duration-only recovery costs, exact deletion-neutral fees, and all other recovery constraints remain unchanged. At the default 1-semitone tolerance, a 0.9-semitone match is now eligible.

The 10-case paired URMP replay has identical per-track TP/FP/FN counts at 50, 100, and 200 ms; primary F1 remains **94.03% overall / 91.09% repeat regions**. The change is therefore retained in production, with cache version 30. These are development-cohort results. 67 focused tests pass, including nondefault tolerances and exact-threshold substitutions.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

shared_root = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").is_file()
    )
)
shared_previous = (
    shared_root / "benchmarks/results/repeat_duration_production_2026-10-01"
)
shared_current = (
    shared_root / "benchmarks/results/repeat_shared_pitch_tolerance_2026-10-01"
)
pd.testing.assert_frame_equal(
    pd.read_csv(shared_previous / "rows.csv"), pd.read_csv(shared_current / "rows.csv")
)
shared_table = pd.read_csv(shared_current / "summary.csv")
display(
    shared_table[shared_table.method.eq("After")][
        ["scope", "tolerance", "tp", "fp", "fn", "f1"]
    ].round(3)
)
print(
    "All per-track counts are unchanged; shared pitch tolerance is the production default."
)


                           scope  tolerance   tp  fp  fn      f1
0                 Repeat regions       0.05  251  44  49  84.370
2                 Repeat regions       0.10  271  24  29  91.092
4                 Repeat regions       0.20  281  14  19  94.454
6   Whole repeat-case recordings       0.05  709  86  87  89.126
8   Whole repeat-case recordings       0.10  748  47  48  94.029
10  Whole repeat-case recordings       0.20  762  33  34  95.789
All per-track counts are unchanged; shared pitch tolerance is the production default.
